# TokenSkip Research Pipeline (Google Colab, Free Tier)
This notebook runs the full HF pipeline for **Qwen2.5-0.5B** (`qwen25_0_5b`) in this repo.

It is configured for Colab free-tier constraints:
- defaults to **mini mode** for fast smoke runs,
- supports **full protocol** if you have enough runtime budget.


In [ ]:
#@title 1) Install dependencies
!pip -q install -U pip
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install transformers datasets peft accelerate sentencepiece llmlingua pyyaml


In [ ]:
#@title 2) Clone/open repo
import os
REPO_URL = "https://github.com/<your-org-or-user>/TokenSkip.git"  # <-- set your repo URL
REPO_DIR = "/content/TokenSkip"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd /content/TokenSkip


In [ ]:
#@title 3) Runtime sanity checks
import torch, platform
print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
#@title 4) Configure run mode and model
# Choose whether to run full protocol or mini smoke test.
RUN_FULL = False  #@param {type:"boolean"}
MODEL_TAG = "qwen25_0_5b"
MODEL_PATH = "Qwen/Qwen2.5-0.5B"

# Training ratio for Phase 1 TokenSkip compressed CoT
TRAIN_RATIO = 0.8  #@param {type:"number"}


In [ ]:
#@title 5) (Optional) adjust protocol settings quickly
# This cell edits protocol.yaml in-place for common Colab tuning.
from pathlib import Path
import yaml

cfg_path = Path("research/configs/protocol.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

# Phase 2 extraction sampling knobs (stochastic, as protocol requires)
cfg.setdefault("phase1", {})
cfg["phase1"]["n_samples"] = 3 if not RUN_FULL else 5
cfg["phase1"]["temperature"] = 1.0
cfg["phase1"]["use_per_step"] = True

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("Updated", cfg_path)


In [ ]:
#@title 6) Phase 0: dataset split (mini or full)
if RUN_FULL:
    !python research/data/split_dataset.py --full
else:
    !python research/data/split_dataset.py --mini


In [ ]:
#@title 7) Phase 1: TokenSkip training (answer-only loss)
!python research/phase1/train.py   --model-type {MODEL_TAG}   --model-path {MODEL_PATH}   --ratio {TRAIN_RATIO}


In [ ]:
#@title 8) Phase 2: extract v_truth (+ per-step vectors)
!python research/phase2/extract_vector.py   --model-type {MODEL_TAG}   --model-path {MODEL_PATH}


In [ ]:
#@title 9) Phase 3: evaluate all 5 conditions
!python research/eval/evaluate_baselines.py   --models {MODEL_TAG}


In [ ]:
#@title 10) Phase 4: aggregate/export tables
!python research/eval/compare_all.py --csv qwen25_0_5b_results.csv --latex


In [ ]:
#@title 11) Quick artifacts check
!find outputs -maxdepth 4 -type f | sed -n '1,200p'
